# Section 2 — H1 Intraday Feature Engineering for Daily Return Forecasting

**Objective.** Predict the *next day's* **continuous (log) percentage return** of
EUR/USD from **H1 (1-hour) bars**. Feeding 24 raw hourly prices per day into a
model would explode the feature dimension and inject microstructure **noise**,
pushing us into the **high-variance / overfitting** regime. Instead we perform
rigorous **Feature Engineering**: each trading day is distilled into a few
domain-motivated statistics, then a **regularised** linear model is fit under a
strictly leak-free, time-series-aware protocol.

This notebook is a *self-contained exploratory study*. It does **not** touch the
production artifacts in `models/`; it mirrors the project's data-sourcing and
no-look-ahead conventions (`src/live_data.py`, `ARCHITECTURE_DOCS.md`) but keeps
the model transparent so every choice is interpretable.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression, RANSACRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (11, 4)

## Task 1 · Feature Engineering (Time-Series Aggregation)

### 1a. Load the raw H1 data

We reuse the project's live-data philosophy (`src/live_data.py`): try a local
**MetaTrader 5** terminal first (deep H1 history), fall back to **Yahoo Finance**
`interval='1h'` (~730 calendar days), then to an on-disk cache so re-runs are
reproducible offline. The result is cached to `../results/eurusd_h1.csv`
(`../` because the notebook runs from `notebooks/`).

In [ ]:
H1_CACHE = os.path.join("..", "results", "eurusd_h1.csv")

def load_h1_data(symbol_mt5="EURUSD", symbol_yf="EURUSD=X", n_bars=60000, cache_path=H1_CACHE):
    """
    Load EUR/USD H1 (1-hour) OHLCV bars.

    Fallback chain mirrors src/live_data.py:
        MetaTrader5 terminal (TIMEFRAME_H1, deep history)
          -> Yahoo Finance interval='1h' (~730 calendar days)
          -> on-disk cache CSV
    Returns (dataframe, source_label).
    """
    # 1) MetaTrader 5 (Windows-only; deepest H1 history)
    try:
        import MetaTrader5 as mt5
        if mt5.initialize():
            rates = mt5.copy_rates_from_pos(symbol_mt5, mt5.TIMEFRAME_H1, 0, n_bars)
            mt5.shutdown()
            if rates is not None and len(rates):
                df = pd.DataFrame(rates)
                df["time"] = pd.to_datetime(df["time"], unit="s")
                df = df.set_index("time").sort_index()
                df = df[["open", "high", "low", "close", "tick_volume"]]
                os.makedirs(os.path.dirname(cache_path), exist_ok=True)
                df.to_csv(cache_path)
                return df, "MT5"
    except Exception:
        pass

    # 2) Yahoo Finance hourly (portable fallback, ~730 days)
    try:
        import yfinance as yf
        h = yf.Ticker(symbol_yf).history(period="730d", interval="1h")
        if h is not None and not h.empty:
            idx = h.index.tz_localize(None) if h.index.tz is not None else h.index
            vol = h["Volume"].astype(float).values if "Volume" in h.columns else 0.0
            df = pd.DataFrame({
                "open":  h["Open"].astype(float).values,
                "high":  h["High"].astype(float).values,
                "low":   h["Low"].astype(float).values,
                "close": h["Close"].astype(float).values,
                "tick_volume": vol,
            }, index=idx)
            df.index.name = "time"
            os.makedirs(os.path.dirname(cache_path), exist_ok=True)
            df.to_csv(cache_path)
            return df, "yfinance"
    except Exception:
        pass

    # 3) On-disk cache (offline reproducibility)
    if os.path.exists(cache_path):
        df = pd.read_csv(cache_path, index_col=0, parse_dates=True)
        return df, "cache"

    raise RuntimeError("No H1 source reachable and no cache present.")

h1, source = load_h1_data()
print(f"Loaded {len(h1):,} H1 bars from: {source}")
print(f"Range: {h1.index.min()}  ->  {h1.index.max()}")
h1.head()

### 1b. Aggregate 24 hourly bars → one daily feature row

**Feature Engineering** is the deliberate act of replacing high-dimensional raw
data with a small set of information-dense, domain-motivated statistics. Each
calendar day's (up to) 24 H1 bars are collapsed into the four requested features,
plus two auxiliary aggregates so the L1/Elastic-Net penalties have redundant
signal to prune (demonstrating feature selection).

We work in **log returns**, $r_t = \ln(P_t / P_{t-1})$, because they are
time-additive and closer to symmetric/stationary than simple returns — the
standard choice for volatility estimation.

| Feature | Definition | Captures |
|---|---|---|
| `Intraday_Volatility` | $\operatorname{std}$ of the day's 24 H1 **log** returns | session turbulence (risk) |
| `Intraday_Momentum` | last H1 close − first H1 open | net directional drift |
| `Daily_Range` | max H1 high − min H1 low | total distance travelled |
| `H1_Moving_Average` | mean of the day's H1 closes | average price level |

> Four informative numbers per day generalise far better than 24×OHLC raw values,
> which would let the model memorise noise — the source of **high variance**.
> *Caveat:* `H1_Moving_Average` and `Daily_Range` sit on the absolute price scale
> (~1.05) and are mildly non-stationary; we keep them as specified and let
> **standardisation + regularisation** damp their influence.

In [ ]:
h1 = h1.copy()
h1["h1_log_return"] = np.log(h1["close"] / h1["close"].shift(1))  # continuous H1 return
h1["date"] = h1.index.normalize()                                # calendar-day bucket

g = h1.groupby("date")   # groups the (up to) 24 hourly bars of each day

daily = pd.DataFrame({
    # --- the four features required by Task 1 ---
    "Intraday_Volatility": g["h1_log_return"].std(),                # std of H1 log returns
    "Intraday_Momentum":   g["close"].last() - g["open"].first(),   # 24th close - 1st open
    "Daily_Range":         g["high"].max() - g["low"].min(),        # max high - min low
    "H1_Moving_Average":   g["close"].mean(),                       # mean of hourly closes
    # --- auxiliary aggregates: give L1 / Elastic-Net something to prune ---
    "H1_Volume_Mean":      g["tick_volume"].mean(),
    "Last_Hour_Return":    g["h1_log_return"].last(),
    "hours_in_day":        g.size(),          # completeness sanity feature (not a model input)
})

# Keep reasonably complete FX weekdays. A full session is ~24 H1 bars; Fridays
# close early and holidays are short, so we require at least a half session.
daily = daily[daily["hours_in_day"] >= 12].copy()
daily.index = pd.to_datetime(daily.index)
daily = daily.sort_index()

print(f"{len(daily)} daily rows built from H1 aggregation")
daily.head()

### 1c. Build the target (strict `shift(-1)`, no look-ahead)

The label is the **next** day's continuous percentage return,

$$ y_t \;=\; 100 \cdot \ln\!\left(\frac{P^{\text{close}}_{t+1}}{P^{\text{close}}_{t}}\right), $$

obtained by `shift(-1)` on the daily log-return series (scaled to **percent** to
match the project convention). Pairing the features of day $t$ with the return
realised on $t{+}1$ guarantees every predictor is known *strictly before* the
target — **no look-ahead bias**. Boundary NaNs are dropped; the target is never
forward/backward-filled.

In [ ]:
# Daily close = the last H1 close of each day, aligned to the filtered days.
daily_close = h1.groupby("date")["close"].last()
daily_close.index = pd.to_datetime(daily_close.index)
daily_close = daily_close.reindex(daily.index)

# Continuous daily return in PERCENT, then shift(-1) so row t holds t+1's return.
daily_log_ret_pct = np.log(daily_close / daily_close.shift(1)) * 100.0
daily["target_return"] = daily_log_ret_pct.shift(-1)

FEATURES = ["Intraday_Volatility", "Intraday_Momentum", "Daily_Range",
            "H1_Moving_Average", "H1_Volume_Mean", "Last_Hour_Return"]

data = daily[FEATURES + ["target_return"]].dropna()   # drops boundary NaNs only
X, y = data[FEATURES], data["target_return"]

print(f"Modelling frame: {X.shape[0]} days x {X.shape[1]} features")
print(f"Target (next-day % log-return): mean={y.mean():+.4f}  std={y.std():.4f}")
X.describe().T[["mean", "std", "min", "max"]]

## Task 2 · Chronological split & regularised modelling

### 2a. Chronological Train / Hold-out split — *never shuffle*

The most recent **20%** of days are locked away as an untouched **hold-out test
set**; the earliest **80%** are used for training and cross-validation. For
time-series data a random split is invalid: it would place *future* days in the
training set and *past* days in validation, leaking information backwards
(**look-ahead bias**) and producing scores that never survive live trading.

In [ ]:
split = int(len(X) * 0.80)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"Train: {X_train.index.min().date()} -> {X_train.index.max().date()}  ({len(X_train)} days)")
print(f"Test : {X_test.index.min().date()} -> {X_test.index.max().date()}  ({len(X_test)} days)")

### 2b. The Bias–Variance Tradeoff and regularised loss functions

The expected test error of an estimator decomposes as

$$ \mathbb{E}\big[(y-\hat f(x))^2\big] \;=\; \underbrace{\big(\text{Bias}[\hat f(x)]\big)^2}_{\text{under-fit}} \;+\; \underbrace{\text{Var}[\hat f(x)]}_{\text{over-fit}} \;+\; \underbrace{\sigma^2_{\varepsilon}}_{\text{irreducible}}. $$

Ordinary least squares minimises only the empirical error and, on noisy FX
features, lands in the **high-variance** corner. **Regularisation** adds a penalty
on the coefficient norm, deliberately injecting a little **bias** to buy a large
reduction in **variance**. Writing the data-fit term as
$\mathcal{L}(\beta)=\tfrac{1}{2n}\lVert y - X\beta \rVert_2^2$:

**Ridge (L2 penalty)** — shrinks all coefficients smoothly, none to exactly zero:

$$ \hat\beta^{\text{Ridge}} = \arg\min_{\beta}\; \mathcal{L}(\beta) + \alpha\,\lVert\beta\rVert_2^2, \qquad \lVert\beta\rVert_2^2 = \sum_{j=1}^{p}\beta_j^2. $$

**Lasso (L1 penalty)** — the non-differentiable corner of the $\ell_1$ ball drives
weak coefficients to *exactly* zero → automatic **feature selection**:

$$ \hat\beta^{\text{Lasso}} = \arg\min_{\beta}\; \mathcal{L}(\beta) + \alpha\,\lVert\beta\rVert_1, \qquad \lVert\beta\rVert_1 = \sum_{j=1}^{p}\lvert\beta_j\rvert. $$

**Elastic Net** — a convex blend of L1 and L2 ($\rho$ = `l1_ratio`), keeping L1's
sparsity while sharing weight among correlated features like L2:

$$ \hat\beta^{\text{EN}} = \arg\min_{\beta}\; \mathcal{L}(\beta) + \alpha\Big(\rho\,\lVert\beta\rVert_1 + \tfrac{1-\rho}{2}\,\lVert\beta\rVert_2^2\Big). $$

**RANSAC** — a *robust* meta-estimator: it repeatedly fits on random subsets,
keeps the consensus of **inliers**, and discards outlier days (fat-tailed FX
shocks) rather than penalising coefficients.

Here $\alpha$ is the **bias–variance dial**: $\alpha\to 0$ recovers high-variance
OLS; large $\alpha$ forces a high-bias near-constant model. We select it by
cross-validation, never on the test set.

> *Implementation note:* scikit-learn's `Ridge` omits the $\tfrac{1}{2n}$ scaling
> on the data term (it minimises $\lVert y-X\beta\rVert_2^2 + \alpha\lVert\beta\rVert_2^2$),
> whereas `Lasso`/`ElasticNet` keep it — so raw $\alpha$ values are not directly
> comparable across the two families.

## Task 3 · Strict time-series validation & hyperparameter tuning

We tune the regularisation strength with **`GridSearchCV`**, scoring by mean
absolute error.

**Why `TimeSeriesSplit`, not `KFold`.** Standard $k$-fold shuffles rows, so a fold
can train on *future* days and validate on *past* ones — **test-set leakage** that
flatters the score. `TimeSeriesSplit` instead grows an **expanding window**: fold
$k$ always trains on an initial contiguous block and validates on the *immediately
following* block, so validation data is always in the future relative to training.
This respects the arrow of time and matches this project's no-random-K-fold
invariant. The `StandardScaler` lives *inside* the pipeline, so it is re-fit on
each training fold and never sees validation-fold statistics.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)          # expanding-window time-series CV
alphas = np.logspace(-4, 2, 25)             # 1e-4 ... 1e+2 : the bias-variance grid

def tune(model, name, extra_grid=None):
    # Scaler inside the pipeline => re-fit per fold => no preprocessing leakage.
    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
    grid = {"model__alpha": alphas}
    if extra_grid:
        grid.update(extra_grid)
    gs = GridSearchCV(pipe, grid, cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1)
    gs.fit(X_train, y_train)
    extra = "".join(f" {k.split('__')[1]}={gs.best_params_[k]}"
                    for k in gs.best_params_ if k != "model__alpha")
    print(f"{name:11s} | best alpha = {gs.best_params_['model__alpha']:.4g}{extra} "
          f"| CV MAE = {-gs.best_score_:.4f}%")
    return gs.best_estimator_, gs

ridge_best, ridge_grid = tune(Ridge(random_state=RANDOM_STATE), "Ridge")
lasso_best, lasso_grid = tune(Lasso(random_state=RANDOM_STATE, max_iter=50000), "Lasso")
enet_best,  enet_grid  = tune(ElasticNet(random_state=RANDOM_STATE, max_iter=50000),
                              "ElasticNet", {"model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]})

### Hold-out evaluation vs a no-skill baseline

The **only** honest score is on the untouched hold-out set. We compare Ridge,
Lasso, Elastic Net and a robust **RANSAC** fit against a `DummyRegressor` that
always predicts the training-set mean return. `R² > 0` means a model genuinely
beats "predict the mean"; on near-efficient daily FX we expect everything to
hover around the baseline.

In [ ]:
def evaluate(model, name):
    pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r2   = r2_score(y_test, pred)
    print(f"{name:16s} | MAE {mae:.4f}% | RMSE {rmse:.4f}% | R2 {r2:+.4f}")
    return mae

# Robust regression on the same standardised features.
ransac = Pipeline([("scaler", StandardScaler()),
                   ("model", RANSACRegressor(estimator=LinearRegression(),
                                             random_state=RANDOM_STATE))]).fit(X_train, y_train)
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)

print("Hold-out performance (lower MAE better; R2>0 beats the mean):")
evaluate(baseline,   "Baseline(mean)")
evaluate(ridge_best, "Ridge")
evaluate(lasso_best, "Lasso")
evaluate(enet_best,  "ElasticNet")
evaluate(ransac,     "RANSAC")

### What did L1 keep? (feature selection + the bias–variance dial)

Left: standardised Lasso coefficients — features driven to exactly `0` were pruned
by the L1 penalty. Right: the CV error as a function of $\alpha$; the minimum is
the sweet spot of the **Bias–Variance Tradeoff** that `GridSearchCV` selected.

In [ ]:
coef = pd.Series(lasso_best.named_steps["model"].coef_, index=FEATURES)
kept = int((coef != 0).sum())
print("Lasso coefficients (standardised features):")
print(coef.sort_values(key=abs, ascending=False).to_string())
print(f"\nL1 kept {kept}/{len(FEATURES)} features; the rest were shrunk to exactly 0.")

fig, ax = plt.subplots(1, 2)
ax[0].bar(coef.index, coef.values)
ax[0].axhline(0, color="k", lw=.6)
ax[0].set_title("Lasso coefficients (L1 selection)")
ax[0].tick_params(axis="x", rotation=90)

ax[1].plot(alphas, -ridge_grid.cv_results_["mean_test_score"], marker="o", ms=3, label="Ridge")
ax[1].plot(alphas, -lasso_grid.cv_results_["mean_test_score"], marker="s", ms=3, label="Lasso")
ax[1].set_xscale("log")
ax[1].set_xlabel(r"$\alpha$ (regularisation strength)")
ax[1].set_ylabel("CV MAE (%)")
ax[1].set_title("Bias-Variance dial")
ax[1].legend()
plt.tight_layout()
plt.show()

## Task 4 · Results & honest interpretation

Daily EUR/USD is **near-efficient**: tomorrow's return is dominated by
unforecastable news, so an honest model's $R^2$ on the hold-out set sits around
**0 (often slightly negative)** and its predictions **shrink toward the ~0% mean**
— precisely the behaviour Ridge/Lasso/Elastic-Net exhibit under a well-chosen
$\alpha$. Beating the predict-the-mean baseline by a wide margin here would signal
**leakage**, not a genuine edge. This mirrors the finding documented for the
production models in `ARCHITECTURE_DOCS.md` §4.2.1.

The deliverable's value is the **method**, not a magic signal:

1. **Feature Engineering** — 24 hourly bars distilled into 4 domain-motivated
   daily statistics (+2 auxiliary), reducing variance at the source.
2. A **leak-free protocol** — chronological hold-out, strict `shift(-1)` target,
   scaler-inside-pipeline, and expanding-window **`TimeSeriesSplit`** CV.
3. **Regularisation** — Ridge (L2), Lasso (L1), Elastic Net and robust RANSAC,
   with $\alpha$ tuned by `GridSearchCV` to sit at the bottom of the
   bias–variance curve, Lasso/Elastic-Net doubling as feature selection.